# S43-a -- fine-tune Qwen3 1.7B on the voice-command training set

Four things you do by hand; everything else runs on its own.

1. **Upload two files here** when this notebook's cells ask for them: nothing --
   the training data comes from Google Drive (next cell tells you the folder),
   and the two things you bring back (`predictions.jsonl` and
   `model-q4_k_m.gguf`) are written to that same Drive folder, not uploaded.
   You DO need `score_port.py` (from `scripts/voice/train/` in the repo) in
   this notebook's own file browser (the folder icon on the left) before
   running the scoring cell -- drag it in, or use the upload button there.
2. **Runtime -> Change runtime type -> T4 GPU**, before running anything.
3. **Runtime -> Run all.** Should be 30 to 60 minutes on a T4 with fp16 (an
   earlier run on bf16 emulation took over six hours -- see the install
   cell). The first few cells print progress in seconds; training is the
   long step. If the session disconnects, run it again -- training resumes
   from its last checkpoint (saved every 50 steps) instead of restarting,
   and a "Run all" after a finished run reloads the saved adapter and skips
   training entirely.
4. **When it finishes**, download `predictions.jsonl` and `model-q4_k_m.gguf`
   from the run folder in Drive (the last cell prints the exact path) and
   bring them, plus the score line the scoring cell printed, back to the
   developer session.

See `scripts/voice/train/README.md` in the repo for the full walkthrough,
including what "good" looks like.


In [ ]:
# ---------------------------------------------------------------------------
# Settings -- every hyperparameter in one place (S43-a brief SS4 step 2).
# ---------------------------------------------------------------------------
BASE_MODEL = "Qwen/Qwen3-1.7B"
SEED = 20260912  # fixed everywhere below -- torch, numpy, random, the trainer

EPOCHS = 3
LR = 2e-4

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Longest training conversation is about 610 tokens (system + user turn +
# the empty think block + the answer); 512 cut the answer off before it
# ever reached the model (F-142). Keep this above the measured maximum
# the data cell prints below.
# S50: the widened prompt (~700 tokens) plus a several of three measured ~940 with
# the served model's tokenizer; 1024 held with ~80 to spare, so 1280 keeps the margin
# honest (memory follows each batch's real lengths, not this cap; the data cell asserts).
# S56: replace/swap/copy and the widened grammar's own longer sentences pushed the
# longest chat row (system + user + assistant text, measured with the served model's
# own /tokenize endpoint against data/voice/colab/train.chat.jsonl) to about 1449
# tokens -- over 1280, so raised to the next 128 above that measurement, 1536.
# S58 (S56-b data brief SS3): group 2's own shapes (adjust/split/headcount/the
# two repeat-day kinds) widened the prompt further and a three-command several
# stayed the longest training row; re-measured the same way against the
# regenerated data/voice/colab/train.chat.jsonl (seed 1, n=7000) at 1757 tokens --
# over 1536, so raised again to the next 128 above that measurement, 1792 (the
# notebook's own chat-template tokenisation, with its extra role/special tokens,
# can only run a little higher than this raw measurement; the data cell below
# asserts the real number against this cap before training is allowed to start).
# S56-c (R-417): the long system prompt (~1,500 tokens) moved to documentation and the
# short one at scripts/voice/train/system_prompt.txt -- ~178 tokens, measured -- is now
# what trains and serves. Re-measured the same way against the regenerated
# data/voice/colab/train.chat.jsonl (short prompt, seed 1, n=7000) at 436 tokens for the
# longest row -- well under 1792, so lowered to the next 128 above that measurement, 512
# (the notebook's own chat-template tokenisation, with its extra role/special tokens, can
# only run a little higher than this raw measurement; the data cell below asserts the
# real number against this cap before training is allowed to start).
MAX_LEN = 512
# S56-c (R-417): the fifth run's training time was spent re-reading a 1,500-token
# system prompt before every example -- R-417 moved to a short trained/served prompt
# (scripts/voice/train/system_prompt.txt) and this run doubles BATCH/halves GRAD_ACCUM
# (same effective 32) now that each step is lighter, plus group_by_length=True in the
# Train cell's SFTConfig below sorts batches by length so less of each step is padding.
BATCH = 16
GRAD_ACCUM = 2  # BATCH * GRAD_ACCUM must be the effective batch the brief asks for (32)
EFFECTIVE_BATCH = BATCH * GRAD_ACCUM
assert EFFECTIVE_BATCH == 32, f"expected an effective batch of 32, got {EFFECTIVE_BATCH}"

DATA_DIR = "/content/drive/MyDrive/scheduler-voice/data"
# Fixed, no timestamp -- a resumed session or a later "Run all" finds the
# SAME checkpoints and the SAME saved adapter instead of starting a fresh,
# empty run folder every time (S43-a Colab finding 4: a 6-hour run cannot
# assume one uninterrupted session, and a finished one should never retrain
# by accident). Delete this folder on Drive to force a genuinely new run.
OUT_DIR = "/content/drive/MyDrive/scheduler-voice/run-qwen3-1.7b"

# F-147: a saved adapter or checkpoint is refused unless its trained-on.json
# matches the CURRENT manifest and train.chat.jsonl hash. Set this True only
# to knowingly reuse an adapter saved before that check existed (no
# trained-on.json next to it at all) -- it never overrides an adapter whose
# trained-on.json says it was trained on DIFFERENT data; that always refuses.
ALLOW_UNSTAMPED_ADAPTER = False

# F-148: EXPECTED_FILES is a placeholder in this SOURCE notebook -- Google
# Drive's own web upload can leave an old same-named file in place instead
# of replacing it (how a stale file gets left behind is not always known --
# do not assume it is a renamed duplicate), so even manifest.json on Drive
# can be the stale copy. Only a fingerprint baked into the notebook
# uploaded fresh each run is trustworthy: `npm run voice:prepare` writes a
# COPY of this notebook to data/voice/colab/train_qwen3.ipynb with this
# placeholder replaced by the CURRENT train.chat.jsonl/heldout.jsonl/
# manifest.json sha256 hashes, row counts, and git sha -- upload THAT copy
# to Colab, never this source file. The data-load cell below refuses to
# run (F-148) while this is still the placeholder.
EXPECTED_FILES = None

print(f"BASE_MODEL={BASE_MODEL}  SEED={SEED}  EPOCHS={EPOCHS}  LR={LR}")
print(f"LoRA: r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}")
print(f"targets={TARGET_MODULES}")
print(f"BATCH={BATCH} x GRAD_ACCUM={GRAD_ACCUM} = effective {EFFECTIVE_BATCH}, MAX_LEN={MAX_LEN}")
print(f"DATA_DIR={DATA_DIR}")
print(f"OUT_DIR={OUT_DIR}")
print(f"ALLOW_UNSTAMPED_ADAPTER={ALLOW_UNSTAMPED_ADAPTER}")
print(f"EXPECTED_FILES={'<placeholder -- run npm run voice:prepare>' if EXPECTED_FILES is None else 'prepared from gitSha=' + EXPECTED_FILES['gitSha']}")


In [ ]:
# ---------------------------------------------------------------------------
# Install -- pinned versions (S43-a brief SS4 step 3). See the report for how
# each version below was chosen: confirmed on PyPI on 12 Sept 2026 unless the
# comment beside it says otherwise.
#
# Plain `subprocess` calls, not `!pip` / `%pip` magics -- this keeps every
# cell here syntactically ordinary Python, which is what lets
# `python -m py_compile` check this notebook's code on a machine with no
# GPU and no `torch` at all (S43-a brief SS2).
# ---------------------------------------------------------------------------
# Colab preinstalls TensorFlow. `transformers` must never import it here --
# `is_tf_available()` (called by things like `transformers.set_seed`) would
# otherwise import TensorFlow and whatever protobuf version it wants, which
# broke `transformers.set_seed` the first time this notebook ran end to
# end. Kept as a guard even now that the requirements file which originally
# caused that conflict is no longer installed below. Repeated at the top of
# the next cell too, so it still holds if that cell is ever run on its own.
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import subprocess
import sys

# Colab preinstalls torchao 0.10.0; peft 0.20's LoRA tuner PROBES for
# torchao the moment `get_peft_model()` runs (peft/tuners/lora/torchao.py's
# `is_torchao_available()`) and refuses anything below 0.16.0 -- even
# though this pipeline never quantises with it. Uninstall it outright,
# before any pip install below; never upgrade it in place, since a newer
# torchao pulls its own torch and risks the same GPU-vs-CPU wheel problem
# the requirements-file fix above already guards against.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)


def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


# llama.cpp has no PyPI package and does not use semantic versioning the
# way the packages below do -- there is no version number to "confirm on
# PyPI". Clone HEAD of the default branch; if `convert_hf_to_gguf.py` ever
# stops recognising Qwen3's architecture, that is a llama.cpp-side
# regression to chase there, not a pin to bump here (brief SS2's honesty
# rule -- said plainly rather than guessing a tag that may not exist).
#
# The clone below is guarded because this cell must survive being run
# twice in the same runtime (F-141).
#
# We do NOT install llama.cpp's own requirements file
# (requirements-convert_hf_to_gguf.txt). It pins `torch==2.11.0` from a CPU
# wheel index and, via its included requirements-convert_legacy_llama.txt,
# `transformers==4.57.6` -- either one breaks a GPU training runtime (the
# second Colab finding was exactly this: `ImportError: cannot import name
# 'GenerationMixin' from 'transformers.generation'`, from the transformers
# downgrade). `convert_hf_to_gguf.py` only actually needs `gguf`,
# `sentencepiece`, `numpy`, `safetensors`, and the torch/transformers this
# cell already installs -- numpy comes from Colab's base image and
# safetensors from transformers' own dependencies, so only the first two
# need installing here, BEFORE our five pins below (same reasoning as
# before: this runs first, our pins below still get the last word).
LLAMA_CPP_DIR = "/content/llama.cpp"
if os.path.exists(os.path.join(LLAMA_CPP_DIR, "convert_hf_to_gguf.py")):
    print("llama.cpp already present at /content/llama.cpp; not cloning again")
else:
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", LLAMA_CPP_DIR],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        raise SystemExit(f"git clone failed ({r.returncode}):\n{r.stderr}")
pip_install("gguf>=0.10", "sentencepiece>=0.1.98,<0.3.0")

# Colab ships its own `torch` build matched to the runtime's CUDA version --
# do not pin or reinstall it here. Installing a mismatched torch wheel is
# the single most common way to break a Colab GPU runtime.
pip_install(
    "transformers==5.17.0",   # confirmed on PyPI 12 Sept 2026; requires torch>=2.5, python>=3.10
    "peft==0.20.0",           # confirmed on PyPI 12 Sept 2026 (released 28 Jul 2026)
    "trl==1.13.0",            # confirmed on PyPI 12 Sept 2026; v1.0 (13 Aug 2026) is the
                               # first stable line -- SFTConfig's `assistant_only_loss` is
                               # this cell's reason for wanting >=1.0, not just a recent patch
    "datasets==5.0.1",        # confirmed on PyPI 12 Sept 2026
    "accelerate==1.15.0",     # confirmed on PyPI 12 Sept 2026
)

import torch

print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached to this runtime. In the Colab menu: "
        "Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all."
    )

# T4 (compute capability 7.5) has no bf16 hardware -- newer torch answers
# `torch.cuda.is_bf16_supported()` True anyway and EMULATES it in software,
# which is why the first real Colab run trained at roughly 0.01 it/s (about
# 100 seconds per step) instead of a normal few it/s, with the loss already
# at its floor by step 80. Decide by compute capability instead: Ampere
# (8.x) and newer have real bf16 hardware; T4, V100 and everything older
# train in fp16. The tokenizer/model cell below reads this to set `DTYPE`,
# which is what the training cell actually keys its precision off of.
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
precision_note = "bf16" if USE_BF16 else "fp16 (a T4 has no bf16 hardware -- this is expected)"
print(f"GPU: {torch.cuda.get_device_name(0)}  precision: {precision_note}")

# Assert our five pins actually won the install order above -- a future
# requirements file (this notebook's own, or one llama.cpp adds later)
# fighting a pin fails HERE, loudly, naming the actual version, not three
# cells later as an unexplained ImportError (brief SS2: "no cell swallows
# an error").
import accelerate
import datasets
import peft
import transformers
import trl

PINNED_VERSIONS = {
    "transformers": (transformers, "5.17.0"),
    "peft": (peft, "0.20.0"),
    "trl": (trl, "1.13.0"),
    "datasets": (datasets, "5.0.1"),
    "accelerate": (accelerate, "1.15.0"),
}
for name, (module, expected) in PINNED_VERSIONS.items():
    actual = module.__version__
    assert actual == expected, (
        f"{name} is {actual}, expected {expected} -- a later install in this cell "
        "(most likely a requirements file with its own pin) downgraded or upgraded it "
        "after our pin ran"
    )
print("pins hold:", ", ".join(f"{n}=={m.__version__}" for n, (m, _) in PINNED_VERSIONS.items()))

# Confirm the torchao uninstall near the top of this cell actually took --
# peft's LoRA tuner probes for torchao lazily, at `get_peft_model()` time,
# so nothing above would have caught it coming back (e.g. pulled in again
# as another package's dependency).
import importlib.util

assert importlib.util.find_spec("torchao") is None, (
    "torchao is still installed. peft's LoRA tuner probes for it at get_peft_model() "
    "time and refuses Colab's version (0.10.0, below peft's 0.16.0 floor); this "
    "pipeline never uses torchao, so re-run the uninstall near the top of this cell "
    "instead of upgrading it."
)


In [ ]:
# ---------------------------------------------------------------------------
# Mount Drive and load data (S43-a brief SS4 step 4).
# ---------------------------------------------------------------------------
# Colab preinstalls TensorFlow; keep it out of this cell too (see the
# install cell's comment -- this repeats the guard so this cell stays safe
# to run on its own).
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import hashlib
import json
import random

import numpy as np
from google.colab import drive


def seed_everything(seed):
    """Seeds python's `random`, `numpy`, and `torch` (CPU and every CUDA
    device) by hand -- NOT `transformers.set_seed`, which calls
    `is_tf_available()` and so imports TensorFlow. No TensorFlow path here,
    on purpose (this is the fix for the first Colab run's
    `ImportError: cannot import name 'runtime_version' from
    'google.protobuf'`, eight frames inside `import tensorflow`)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)

drive.mount("/content/drive")

TRAIN_CHAT_PATH = os.path.join(DATA_DIR, "train.chat.jsonl")
HELDOUT_PATH = os.path.join(DATA_DIR, "heldout.jsonl")
MANIFEST_PATH = os.path.join(DATA_DIR, "manifest.json")

for path in (TRAIN_CHAT_PATH, HELDOUT_PATH, MANIFEST_PATH):
    if not os.path.exists(path):
        raise SystemExit(
            f"{path} is missing. Upload the three files `npm run voice:prepare` wrote "
            f"(train.chat.jsonl, heldout.jsonl, manifest.json) to {DATA_DIR} in Drive, "
            "then re-run this cell."
        )

if EXPECTED_FILES is None:
    raise SystemExit(
        "F-148: this notebook has not been prepared -- EXPECTED_FILES in the Settings "
        "cell is still the placeholder. Run `npm run voice:prepare` on this machine, "
        "then upload the COPY it writes, data/voice/colab/train_qwen3.ipynb (not this "
        "source file), to Colab instead."
    )


def _sha256_file(path):
    """Streamed sha256 (F-148) -- these files can run to several MB; read
    in fixed chunks rather than loading the whole file into memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _count_jsonl_rows(path):
    with open(path, "r", encoding="utf-8") as fh:
        return sum(1 for line in fh if line.strip())


# F-148: an old same-named file can be left on Drive instead of the one
# this run actually prepared -- how is not always known (Drive's web
# upload can leave one in place, but that is not confirmed to be the only
# way it happens). Checking the data files against manifest.json alone is
# not enough for that reason -- this checks every file on Drive against
# EXPECTED_FILES, the fingerprint baked into THIS notebook by
# `npm run voice:prepare` when it wrote the copy that was uploaded, before
# any of the three files is trusted.
print(f"this notebook was prepared from git sha {EXPECTED_FILES['gitSha']}")
_expected_by_file = {
    "train.chat.jsonl": (TRAIN_CHAT_PATH, EXPECTED_FILES["train.chat.jsonl"]),
    "heldout.jsonl": (HELDOUT_PATH, EXPECTED_FILES["heldout.jsonl"]),
    "manifest.json": (MANIFEST_PATH, EXPECTED_FILES["manifest.json"]),
}
for _filename, (_path, _expected) in _expected_by_file.items():
    _actual_sha256 = _sha256_file(_path)
    _actual_rows = _count_jsonl_rows(_path) if "rows" in _expected else None
    _mismatch = _actual_sha256 != _expected["sha256"] or (
        _actual_rows is not None and _actual_rows != _expected["rows"]
    )
    if _mismatch:
        print(f"F-148: {_filename} on Drive is not the file this notebook expects")
        print(f"  expected sha256: {_expected['sha256']}")
        print(f"  Drive sha256:    {_actual_sha256}")
        if _actual_rows is not None:
            print(f"  expected rows:   {_expected['rows']}")
            print(f"  Drive rows:      {_actual_rows}")
        print(
            f"However the old file got there, delete every copy of {_filename} from "
            "this folder on Drive (Drive's web upload can keep an old same-named "
            f"file), then upload the current data/voice/colab/{_filename} again and "
            "check its size and date in the folder."
        )
        raise SystemExit(f"F-148: {_filename} on Drive is not the file this notebook expects")
    print(f"{_filename} matches this notebook's expected fingerprint")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_chat_rows = load_jsonl(TRAIN_CHAT_PATH)
heldout_rows = load_jsonl(HELDOUT_PATH)

assert len(train_chat_rows) == manifest["trainRows"], (
    f"train.chat.jsonl has {len(train_chat_rows)} rows, manifest.json says {manifest['trainRows']}"
)
assert len(heldout_rows) == manifest["heldoutRows"], (
    f"heldout.jsonl has {len(heldout_rows)} rows, manifest.json says {manifest['heldoutRows']}"
)

train_sentences = {row["messages"][1]["content"] for row in train_chat_rows}
heldout_sentences = {row["sentence"] for row in heldout_rows}
overlap = train_sentences & heldout_sentences
assert not overlap, (
    f"{len(overlap)} sentence(s) appear in BOTH train and held-out, e.g. {next(iter(overlap))!r} "
    "-- the model would be scored on something it trained on"
)

SYSTEM_PROMPT = manifest["systemPrompt"]  # read from the manifest, never retyped here

print(f"train: {len(train_chat_rows)} chat rows   held-out: {len(heldout_rows)} rows")
print(f"git sha this data was generated from: {manifest['gitSha']}")
print(
    "rule-parser baseline (from manifest.json): "
    f"clean {manifest['ruleParserBaseline']['clean'] * 100:.1f}%  "
    f"perturbed {manifest['ruleParserBaseline']['perturbed'] * 100:.1f}%"
)


In [ ]:
# ---------------------------------------------------------------------------
# Tokenizer and base model (S43-a brief SS4 step 5).
# ---------------------------------------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

DTYPE = torch.bfloat16 if USE_BF16 else torch.float16  # USE_BF16 decided in the install cell
print(f"loading {BASE_MODEL} in {DTYPE}")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map="auto")
base_model = model  # kept so the training cell can reload a saved adapter onto
# the ORIGINAL base model if one already exists on Drive, bypassing the LoRA
# cell's fresh (untrained) wrap below (S43-a Colab finding 4)

# Thinking mode is OFF in both training and inference (S43-a brief SS2) --
# `enable_thinking=False` on every `apply_chat_template` call in this
# notebook, this one included.
#
# Assert the chat template's own round trip against the manifest's sample
# BEFORE spending an hour training on top of it (brief SS2: "asserts its own
# serialisation against a sample in the manifest").
sample = manifest["sample"]
sample_row = next(r for r in heldout_rows if r["id"] == sample["id"])
assert sample_row["sentence"] == sample["sentence"], "manifest sample does not match heldout.jsonl"

sample_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": sample_row["sentence"]},
]
prompt_text = tokenizer.apply_chat_template(
    sample_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
full_text = prompt_text + sample["canonicalForm"]
round_trip_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
decoded = tokenizer.decode(round_trip_ids)
assert sample["canonicalForm"] in decoded, (
    "the tokenizer did not round-trip the manifest's sample form byte for byte -- "
    "the chat template or a special token changed underneath this notebook"
)
print(f"chat-template round trip OK against manifest sample {sample['id']!r}")

# Measure every training conversation at full length, the way the trainer
# will actually see it (SFTConfig(max_length=MAX_LEN) truncates at this same
# tokenisation) -- and refuse to proceed if MAX_LEN would cut any answer off
# again (F-142: MAX_LEN=512 silently truncated the answer out of every row).
# return_dict=False pins the return shape to a plain token list in both
# transformers 4.x and 5.x (5.x otherwise defaults to a BatchEncoding, a
# UserDict -- not a dict subclass -- so an isinstance(encoded, dict) guard
# misses it and len(encoded) silently becomes a key count, not a token
# count); the shape guard below and the floor assertion after the loop
# catch it anyway if the return shape is ever wrong.
conversation_lengths = []
for row in train_chat_rows:
    encoded = tokenizer.apply_chat_template(
        row["messages"], tokenize=True, add_generation_prompt=False, return_dict=False
    )
    if not isinstance(encoded, list):
        encoded = encoded["input_ids"]
    conversation_lengths.append(len(encoded))

conversation_lengths.sort()
median_len = conversation_lengths[len(conversation_lengths) // 2]
max_len_seen = conversation_lengths[-1]
print(
    f"training conversations: median {median_len} tokens, longest {max_len_seen} tokens, "
    f"MAX_LEN={MAX_LEN}"
)
# R-417 (15 Sept): the prompt is ~200 tokens now, so a median conversation is ~300; the
# floor that proves the tokeniser returned a token list is the prompt plus a sentence and
# a form, well above 150 -- it was 300 when the prompt alone was 900 tokens (F-142).
assert median_len > 150, (
    f"the tokeniser returned {median_len} for a median conversation; a full conversation "
    "is at least the prompt plus a sentence and a form, so this is not a token list (F-142)"
)
assert max_len_seen < MAX_LEN, (
    f"F-142: the longest training conversation is {max_len_seen} tokens, at or above "
    f"MAX_LEN={MAX_LEN} -- the trainer would truncate the answer off the end of that row, "
    "the same bug that scored product at 2.5%. Raise MAX_LEN."
)


In [ ]:
# ---------------------------------------------------------------------------
# LoRA (S43-a brief SS4 step 6).
# ---------------------------------------------------------------------------
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# ---------------------------------------------------------------------------
# Train (S43-a brief SS4 step 7).
#
# Loss on the assistant turn only, via TRL 1.x's `SFTConfig(assistant_only_loss=True)`
# -- this is the "say which API" the brief asks for. TRL patches the chat
# template to add `{% generation %}` markers for known model families
# (Qwen3 included) when this flag is set, so the system and user tokens are
# masked out of the loss without a hand-written response template or a
# `DataCollatorForCompletionOnlyLM` (the older, string-matching approach --
# not used here because it is easy to get subtly wrong across tokenizers,
# per TRL's own docs).
#
# Checkpointed every 50 steps (2 kept) so a disconnect resumes instead of
# restarting a run that can take hours (S43-a Colab finding 4); and skipped
# outright if a trained adapter is already sitting in ADAPTER_DIR on Drive,
# so a later "Run all" does not retrain for hours by accident -- delete
# ADAPTER_DIR on Drive to force a fresh run.
#
# F-147: the fifth Colab run found the FOURTH run's adapter this way and
# skipped straight past it onto the fifth run's own held-out data -- an
# adapter has the same "resume by existence" hole F-145 closed for
# predictions.jsonl, but nothing said which data it was trained on. Both
# ADAPTER_DIR and CHECKPOINTS_DIR now carry a `trained-on.json` stamp (the
# manifest's own gitSha/trainRows/heldoutRows/seeds, plus train.chat.jsonl's
# and heldout.jsonl's sha256 (F-148: taken from manifest["files"], not
# re-hashed here, since the data-load cell above already checked Drive's
# copies of both files against this notebook's own EXPECTED_FILES) --
# written the moment training starts or finishes, and
# both are refused -- `raise SystemExit`, before anything is skipped or
# resumed -- when the stamp is missing or does not match the CURRENT data.
# ---------------------------------------------------------------------------

from datasets import Dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

ADAPTER_DIR = os.path.join(OUT_DIR, "adapter")
CHECKPOINTS_DIR = os.path.join(OUT_DIR, "checkpoints")


def find_latest_checkpoint(checkpoints_dir):
    """Returns the newest `checkpoint-<step>` folder under `checkpoints_dir`,
    or None if there isn't one yet -- lets a disconnected run resume instead
    of restarting from step 0."""
    if not os.path.isdir(checkpoints_dir):
        return None
    checkpoints = [d for d in os.listdir(checkpoints_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda d: int(d.split("-")[1]))
    return os.path.join(checkpoints_dir, checkpoints[-1])


def _current_stamp():
    """The identifying fields `trained-on.json` records (F-147): the
    manifest's own gitSha/trainRows/heldoutRows/seeds -- read from
    manifest.json, never retyped (see scripts/voice/train/prepare.mjs for
    where these keys come from) -- plus train.chat.jsonl's and
    heldout.jsonl's sha256 (F-148: an adapter now also names the held-out
    set it was scored against). Both hashes come from manifest["files"],
    already verified against Drive by the data-load cell above -- there is
    exactly one place in this notebook that computes a hash."""
    return {
        "gitSha": manifest["gitSha"],
        "trainRows": manifest["trainRows"],
        "heldoutRows": manifest["heldoutRows"],
        "seeds": manifest["seeds"],
        "trainChatSha256": manifest["files"]["train.chat.jsonl"]["sha256"],
        "heldoutSha256": manifest["files"]["heldout.jsonl"]["sha256"],
    }


def _write_stamp(dir_path, stamp):
    os.makedirs(dir_path, exist_ok=True)
    with open(os.path.join(dir_path, "trained-on.json"), "w", encoding="utf-8") as f:
        json.dump(stamp, f, indent=2)
        f.write("\n")


def _read_stamp(dir_path):
    stamp_path = os.path.join(dir_path, "trained-on.json")
    if not os.path.exists(stamp_path):
        return None
    with open(stamp_path, "r", encoding="utf-8") as f:
        return json.load(f)


def _mismatched_fields(saved_stamp, current_stamp):
    """Every key where `saved_stamp` (the adapter's or checkpoint's own
    trained-on.json) differs from `current_stamp` (this run's own data) --
    F-147 prints these field by field rather than just saying "they differ"."""
    return [key for key in current_stamp if saved_stamp.get(key) != current_stamp[key]]


def _print_mismatch(label, dir_path, saved_stamp, current_stamp):
    print(f"F-147: {dir_path}'s trained-on.json does not match the current data:")
    for key in _mismatched_fields(saved_stamp, current_stamp):
        print(f"  {key}:")
        print(f"    {label} says:  {saved_stamp.get(key)!r}")
        print(f"    current data: {current_stamp[key]!r}")


current_stamp = _current_stamp()

if os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    saved_adapter_stamp = _read_stamp(ADAPTER_DIR)
    if saved_adapter_stamp is None and not ALLOW_UNSTAMPED_ADAPTER:
        print(
            f"F-147: {ADAPTER_DIR} has no trained-on.json -- this adapter carries no "
            "record of the data it was trained on, so it cannot be trusted to match the "
            "data this run prepared.\n"
            f"Delete {ADAPTER_DIR} to force a fresh run, or, to knowingly reuse this "
            "adapter as is, set ALLOW_UNSTAMPED_ADAPTER = True in the Settings cell and "
            "run this cell again."
        )
        raise SystemExit(f"F-147: {ADAPTER_DIR} has no trained-on.json")
    elif saved_adapter_stamp is None:
        print(
            f"F-147: {ADAPTER_DIR} has no trained-on.json, but ALLOW_UNSTAMPED_ADAPTER is "
            "True -- reusing it anyway, unverified against the current data"
        )
    else:
        mismatched = _mismatched_fields(saved_adapter_stamp, current_stamp)
        if mismatched:
            _print_mismatch("adapter", ADAPTER_DIR, saved_adapter_stamp, current_stamp)
            raise SystemExit(
                f"F-147: {ADAPTER_DIR} was trained on different data than this run "
                "prepared -- delete it to retrain, or retrain on the SAME data it "
                "matches. No flag overrides a mismatch."
            )
        print(f"training skipped: a trained adapter already exists at {ADAPTER_DIR}")
        print("delete that folder on Drive to retrain from scratch")
        print(
            f"  matches: gitSha={current_stamp['gitSha']} trainRows={current_stamp['trainRows']} "
            f"heldoutRows={current_stamp['heldoutRows']} seeds={current_stamp['seeds']}"
        )
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
else:
    train_dataset = Dataset.from_list([{"messages": row["messages"]} for row in train_chat_rows])

    sft_config = SFTConfig(
        output_dir=CHECKPOINTS_DIR,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_length=MAX_LEN,
        group_by_length=True,  # S56-c (R-417): sort batches by length -- less padding
        bf16=(DTYPE == torch.bfloat16),
        fp16=(DTYPE == torch.float16),
        seed=SEED,
        data_seed=SEED,
        logging_steps=20,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        report_to=[],
        assistant_only_loss=True,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        processing_class=tokenizer,
    )

    resume_from = find_latest_checkpoint(CHECKPOINTS_DIR)
    if resume_from:
        saved_checkpoint_stamp = _read_stamp(CHECKPOINTS_DIR)
        if saved_checkpoint_stamp is None:
            print(
                f"F-147: {CHECKPOINTS_DIR} has a checkpoint but no trained-on.json -- a "
                "half-finished run's checkpoint carries no record of the data it started "
                f"training on.\nDelete {CHECKPOINTS_DIR} to start a fresh run."
            )
            raise SystemExit(f"F-147: {CHECKPOINTS_DIR} has no trained-on.json")
        checkpoint_mismatched = _mismatched_fields(saved_checkpoint_stamp, current_stamp)
        if checkpoint_mismatched:
            _print_mismatch("checkpoint", CHECKPOINTS_DIR, saved_checkpoint_stamp, current_stamp)
            raise SystemExit(
                f"F-147: {CHECKPOINTS_DIR} was trained on different data than this run "
                "prepared -- delete it to retrain, or retrain on the SAME data it "
                "matches. No flag overrides a mismatch."
            )
        print(f"resuming from checkpoint: {resume_from}")
    else:
        print("starting training from scratch")
        _write_stamp(CHECKPOINTS_DIR, current_stamp)

    train_result = trainer.train(resume_from_checkpoint=resume_from)
    print(train_result.metrics)

    os.makedirs(ADAPTER_DIR, exist_ok=True)
    trainer.save_model(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    _write_stamp(ADAPTER_DIR, current_stamp)
    print(f"adapter saved to {ADAPTER_DIR}")


In [ ]:
# ---------------------------------------------------------------------------
# Predict on held-out (S43-a brief SS4 step 8).
#
# Greedy decoding; generation is stopped the moment the text generated SO
# FAR closes a complete top-level JSON object (brief SS2: "the generation is
# stopped at the first closing brace of a complete JSON object") -- a
# `StoppingCriteria` that decodes the tokens generated so far on every step
# and counts braces, skipping over quoted strings so a brace inside a name
# never miscounts.
# ---------------------------------------------------------------------------
from transformers import StoppingCriteria, StoppingCriteriaList


def _scan_braces(text):
    """Returns (started, depth, closed_at) for `text`: whether a top-level
    '{' has been seen, the current brace depth, and the index one past a
    complete top-level object's closing '}' if one was found."""
    depth = 0
    started = False
    in_string = False
    escaped = False
    for i, ch in enumerate(text):
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
            started = True
        elif ch == "}":
            depth -= 1
            if started and depth == 0:
                return started, depth, i + 1
    return started, depth, None


class JsonObjectStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.done = False

    def __call__(self, input_ids, scores, **kwargs):
        if self.done:
            return True
        generated_ids = input_ids[0][self.prompt_len :]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        _, _, closed_at = _scan_braces(text)
        if closed_at is not None:
            self.done = True
            return True
        return False


def extract_first_json_object(text):
    _, _, closed_at = _scan_braces(text)
    if closed_at is None:
        return None
    start = text.find("{")
    return text[start:closed_at] if start != -1 else None


import time

model.eval()

PREDICTIONS_PATH = os.path.join(OUT_DIR, "predictions.jsonl")

# Resume support (S43-a Colab finding 5): Colab's free tier has a two-hour
# session deadline, and 400 rows at an unlucky rows/s can outlast it. If
# predictions.jsonl already has rows for some ids, skip those and continue
# appending instead of starting over.
done_ids = set()
n_failed = 0
# F-145: the maintainer's fourth Colab run resumed predictions.jsonl (by
# id) against a held-out file that had been silently regenerated under
# it -- 400 single-row predictions answered sentences that were no longer
# in data/voice/heldout.jsonl, and the scorer, joining by id, had no way
# to notice. Before any id is treated as already done, every existing row
# is checked against the CURRENT heldout_rows by id: a row with no
# "sentence" field (a predictions.jsonl from before this change) or a
# "sentence" that does not match the held-out row of the same id stops
# this cell outright, before anything is skipped.
heldout_sentence_by_id = {row["id"]: row["sentence"] for row in heldout_rows}
if os.path.exists(PREDICTIONS_PATH):
    with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            existing = json.loads(line)
            existing_id = existing["id"]
            existing_sentence = existing.get("sentence")
            heldout_sentence = heldout_sentence_by_id.get(existing_id)
            # Three distinct reasons a row cannot be trusted to skip its id,
            # checked in this order so the message never claims a sentence
            # "differs" when there was no held-out sentence to compare against
            # at all (a stale id would otherwise print "held-out sentence: None"
            # as though that None were a real, differing sentence).
            if existing_id not in heldout_sentence_by_id:
                reason = "is not in the current held-out file"
            elif existing_sentence is None:
                reason = 'carries no "sentence" field (a predictions.jsonl from before F-145)'
            elif existing_sentence != heldout_sentence:
                reason = "answers a different sentence than the current held-out file"
            else:
                reason = None
            if reason is not None:
                raise RuntimeError(
                    f'F-145: predictions.jsonl row for id {existing_id!r} {reason} -- '
                    "these predictions answered a different held-out file.\n"
                    f"  predictions.jsonl sentence:        {existing_sentence!r}\n"
                    f"  data/voice/heldout.jsonl sentence: {heldout_sentence!r}\n"
                    "Delete predictions.jsonl and run this cell again."
                )
            done_ids.add(existing_id)
            if existing.get("form") is None:
                n_failed += 1
if done_ids:
    print(f"resuming: {len(done_ids)} row(s) already predicted, skipping them")

rows_to_predict = [row for row in heldout_rows if row["id"] not in done_ids]
total = len(heldout_rows)

if not rows_to_predict:
    print(f"all {total} rows already predicted -- nothing to do")
else:
    start_time = time.time()
    with open(PREDICTIONS_PATH, "a", encoding="utf-8") as out_file:
        for i, row in enumerate(rows_to_predict):
            row_start = time.time()
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["sentence"]},
            ]
            prompt_text = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
            prompt_len = inputs["input_ids"].shape[1]
            stopper = JsonObjectStoppingCriteria(tokenizer, prompt_len)
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                    num_beams=1,
                    stopping_criteria=StoppingCriteriaList([stopper]),
                    pad_token_id=tokenizer.pad_token_id,
                )

            if i == 0:
                # Timed alone and printed immediately -- an emulated-precision
                # slow path (S43-a Colab finding 4) is visible within seconds,
                # not after sitting on blank output for the whole cell.
                generated_tokens = output_ids.shape[1] - prompt_len
                print(f"first row: {time.time() - row_start:.1f}s for {generated_tokens} tokens")

            generated_text = tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
            json_text = extract_first_json_object(generated_text)
            form = None
            if json_text is not None:
                try:
                    form = json.loads(json_text)
                except json.JSONDecodeError:
                    form = None
            if form is None:
                n_failed += 1

            out_file.write(
                json.dumps({"id": row["id"], "sentence": row["sentence"], "form": form})
                + "\n"
            )
            out_file.flush()

            done_this_run = i + 1
            if done_this_run % 25 == 0 or done_this_run == len(rows_to_predict):
                elapsed = time.time() - start_time
                rate = done_this_run / elapsed if elapsed > 0 else 0.0
                print(
                    f"{len(done_ids) + done_this_run}/{total} rows, {elapsed:.0f}s, "
                    f"{rate:.2f} rows/s, {n_failed} failed to parse"
                )

    print(f"finished this run: {len(rows_to_predict)} row(s) predicted, {n_failed} failed to parse total")

print(f"predictions for all {total} row(s) are in {PREDICTIONS_PATH}")


In [ ]:
# ---------------------------------------------------------------------------
# Score (S43-a brief SS4 step 9).
#
# `score_port.py` (from `scripts/voice/train/` in the repo) must already be
# in this notebook's own file browser (the folder icon on the left) --
# step 1 of the title cell above. This table is for your eyes in Colab only;
# the run that counts is `npm run voice:score`-shaped, on the developer's
# machine, against THIS predictions.jsonl (brief SS2).
# ---------------------------------------------------------------------------
import importlib.util

SCORE_PORT_PATH = "/content/score_port.py"
if not os.path.exists(SCORE_PORT_PATH):
    raise FileNotFoundError(
        "score_port.py is missing from /content/. Upload it (from "
        "scripts/voice/train/score_port.py in the repo) using the Files pane "
        "on the left, then re-run this cell."
    )

spec = importlib.util.spec_from_file_location("score_port", SCORE_PORT_PATH)
score_port = importlib.util.module_from_spec(spec)
spec.loader.exec_module(score_port)

result = score_port.score_predictions(
    heldout_rows, score_port.load_jsonl(PREDICTIONS_PATH)
)
score_port.print_table(result)

clean_rate = score_port.rate(result["clean"])
print(f"clean rate {clean_rate * 100:.1f}%")
print(
    "rule-parser baseline (from manifest.json): "
    f"clean {manifest['ruleParserBaseline']['clean'] * 100:.1f}%  "
    f"perturbed {manifest['ruleParserBaseline']['perturbed'] * 100:.1f}%"
)
print(
    "This is Colab's own read. The number that goes on S43's card is "
    "`node scripts/voice/score.mjs --heldout data/voice/heldout.jsonl "
    "--predictions predictions.jsonl --bar 0.95`, run on the developer's machine."
)


In [ ]:
# ---------------------------------------------------------------------------
# Merge and export (S43-a brief SS4 step 10).
#
# `convert_hf_to_gguf.py` below runs inside this same training environment,
# against transformers 5.17 (not the 4.57.6 its own requirements file
# wants -- see the install cell). If it ever objects to that version, the
# fix is a separate venv for this converter step, never a downgrade here.
# ---------------------------------------------------------------------------
MERGED_DIR = os.path.join(OUT_DIR, "merged")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"merged model saved to {MERGED_DIR}")

GGUF_F16_PATH = os.path.join(OUT_DIR, "model-f16.gguf")
subprocess.run(
    [
        sys.executable,
        "/content/llama.cpp/convert_hf_to_gguf.py",
        MERGED_DIR,
        "--outfile",
        GGUF_F16_PATH,
        "--outtype",
        "f16",
    ],
    check=True,
)

# llama.cpp ships no prebuilt `llama-quantize` via pip -- build it.
subprocess.run(["cmake", "-B", "/content/llama.cpp/build", "/content/llama.cpp"], check=True)
subprocess.run(
    ["cmake", "--build", "/content/llama.cpp/build", "--target", "llama-quantize", "-j", "4"],
    check=True,
)

GGUF_Q4_PATH = os.path.join(OUT_DIR, "model-q4_k_m.gguf")
subprocess.run(
    ["/content/llama.cpp/build/bin/llama-quantize", GGUF_F16_PATH, GGUF_Q4_PATH, "Q4_K_M"],
    check=True,
)

f16_size = os.path.getsize(GGUF_F16_PATH)
q4_size = os.path.getsize(GGUF_Q4_PATH)
print(f"f16:    {GGUF_F16_PATH}  ({f16_size / 1e9:.2f} GB)")
print(f"Q4_K_M: {GGUF_Q4_PATH}  ({q4_size / 1e9:.2f} GB)")


## What to bring back

From `{OUT_DIR}` in Drive (the exact path was printed by the Settings cell
above -- it is fixed, the same on every run):

- `predictions.jsonl`
- `model-q4_k_m.gguf`

Download both into `data/voice/runs/<timestamp>/` on your machine (that
folder is gitignored), plus the score line the scoring cell printed, and
bring them to the developer session -- see `scripts/voice/train/README.md`
step 5 for the exact scoring command that goes on S43's card.
